# 2주차 4차시 실습 — 데이터 품질 검증 및 시각화

## 실습 목표
- 전처리된 데이터의 품질을 자동으로 검사
- 필수 컬럼, 결측치, 중복, 숫자형 여부, 허용 범위 검사
- IQR 기반 이상치 탐지
- 품질검사 결과를 표와 그래프로 시각화
- 최종적으로 PASS / WARN / FAIL 형태로 결과 요약

```text
students_clean.csv
        ↓
Schema Check
Missing Check
Duplicate Check
Type Check
Range Check
IQR Outlier Check
        ↓
Quality Report
        ↓
표 + 그래프
        ↓
PASS / WARN / FAIL
```

## 셀 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 셀 2. CSV 파일 업로드

2차시에서 생성한 `students_clean.csv` 파일을 업로드합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
FILE_PATH = "students_clean.csv"
df = pd.read_csv(FILE_PATH)
print("데이터 크기:", df.shape)
display(df.head())

## 셀 3. 데이터 전체 확인

In [ ]:
print("=== 데이터 크기 ===")
print(df.shape)
print("\n=== 컬럼 ===")
print(df.columns.tolist())
print("\n=== 자료형 ===")
print(df.dtypes)
print("\n=== 기초 통계량 ===")
display(df.describe(include="all"))

## 셀 4. 품질검증 규칙 정의
- `REQUIRED_COLUMNS`: 반드시 존재해야 하는 컬럼
- `RANGE_RULES`: 각 숫자형 컬럼의 허용 범위

In [ ]:
REQUIRED_COLUMNS = [
    "student_id", "study_hours", "attendance", "assignment_score",
    "midterm_score", "avg_score", "risk_flag", "outlier_flag",
]

RANGE_RULES = {
    "study_hours": (0, 12),
    "attendance": (0, 100),
    "assignment_score": (0, 100),
    "midterm_score": (0, 100),
    "avg_score": (0, 100),
    "risk_flag": (0, 1),
    "outlier_flag": (0, 1),
}

## 셀 5. 필수 컬럼 검사

In [ ]:
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    print("[FAIL] 필수 컬럼 누락")
    print(missing_columns)
else:
    print("[PASS] 모든 필수 컬럼이 존재합니다.")

## 셀 6. 결측치 검사

In [ ]:
missing_report = df.isna().sum()
missing_report = pd.DataFrame({
    "column": missing_report.index,
    "missing_count": missing_report.values
})
display(missing_report)

total_missing = int(df.isna().sum().sum())
print("전체 결측치:", total_missing)
if total_missing == 0:
    print("[PASS] 결측치 없음")
else:
    print(f"[FAIL] 결측치 {total_missing}건 존재")

## 셀 7. 결측치 그래프

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(missing_report["column"], missing_report["missing_count"])
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing Count")
plt.xticks(rotation=45)
plt.show()

## 셀 8. student_id 중복 검사

In [ ]:
if "student_id" in df.columns:
    duplicate_count = int(df["student_id"].duplicated().sum())
    print("student_id 중복:", duplicate_count)
    if duplicate_count == 0:
        print("[PASS] student_id 중복 없음")
    else:
        print(f"[FAIL] student_id 중복 {duplicate_count}건")
    duplicates = df[df["student_id"].duplicated(keep=False)]
    display(duplicates)
else:
    duplicate_count = 0
    print("[FAIL] student_id 컬럼이 없습니다.")

## 셀 9. 숫자형 데이터 검사

In [ ]:
type_results = []
for col in RANGE_RULES:
    if col not in df.columns:
        continue
    numeric = pd.to_numeric(df[col], errors="coerce")
    non_numeric = numeric.isna() & df[col].notna()
    type_results.append({
        "column": col,
        "non_numeric_count": int(non_numeric.sum())
    })
type_report = pd.DataFrame(type_results)
display(type_report)

## 셀 10. 허용 범위 검사

In [ ]:
range_results = []
for col, (min_value, max_value) in RANGE_RULES.items():
    if col not in df.columns:
        continue
    numeric = pd.to_numeric(df[col], errors="coerce")
    invalid = (numeric < min_value) | (numeric > max_value)
    range_results.append({
        "column": col,
        "min": min_value,
        "max": max_value,
        "violation_count": int(invalid.sum())
    })
range_report = pd.DataFrame(range_results)
display(range_report)

## 셀 11. 범위 위반 그래프

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(range_report["column"], range_report["violation_count"])
plt.title("Range Violations by Column")
plt.xlabel("Column")
plt.ylabel("Violation Count")
plt.xticks(rotation=45)
plt.show()

## 셀 12. 범위 위반 데이터 직접 확인

In [ ]:
for col, (min_value, max_value) in RANGE_RULES.items():
    if col not in df.columns:
        continue
    numeric = pd.to_numeric(df[col], errors="coerce")
    invalid = df[(numeric < min_value) | (numeric > max_value)]
    if len(invalid) > 0:
        print(f"\n=== {col} 범위 위반 ===")
        cols_to_show = [c for c in ["student_id", col] if c in invalid.columns]
        display(invalid[cols_to_show])

## 셀 13. IQR 이상치 탐지 함수

```text
IQR = Q3 - Q1
Lower = Q1 - 1.5 × IQR
Upper = Q3 + 1.5 × IQR
```

In [ ]:
def check_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

## 셀 14. IQR 이상치 검사

In [ ]:
iqr_results = []
IQR_COLUMNS = ["study_hours", "avg_score"]
for col in IQR_COLUMNS:
    if col not in df.columns:
        continue
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    outliers, lower, upper = check_iqr_outliers(series)
    iqr_results.append({
        "column": col,
        "lower": round(lower, 2),
        "upper": round(upper, 2),
        "outlier_count": len(outliers)
    })
iqr_report = pd.DataFrame(iqr_results)
display(iqr_report)

## 셀 15. IQR 이상치 그래프
- 범위 위반: FAIL
- IQR 이상치: WARN

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(iqr_report["column"], iqr_report["outlier_count"])
plt.title("IQR Outliers by Column")
plt.xlabel("Column")
plt.ylabel("Outlier Count")
plt.show()

series = pd.to_numeric(
    df["study_hours"],
    errors="coerce"
)

q1 = series.quantile(0.25)
q3 = series.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (
    (series < lower) |
    (series > upper)
)

study_hours_outliers = df.loc[
    mask,
    ["student_id", "study_hours"]
]

print(f"Q1 = {q1:.2f}")
print(f"Q3 = {q3:.2f}")
print(f"IQR = {iqr:.2f}")
print(f"정상 범위 = {lower:.2f} ~ {upper:.2f}")
print(f"이상치 개수 = {len(study_hours_outliers)}")

display(study_hours_outliers)

## 셀 16. Boxplot으로 이상치 확인

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(pd.to_numeric(df["study_hours"], errors="coerce").dropna(), vert=False)
plt.title("Study Hours - Boxplot")
plt.xlabel("Study Hours")
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(pd.to_numeric(df["avg_score"], errors="coerce").dropna(), vert=False)
plt.title("Average Score - Boxplot")
plt.xlabel("Average Score")
plt.show()

## 셀 17. 품질검사 결과 종합

In [ ]:
quality_summary = []
quality_summary.append({
    "check": "Required Columns",
    "issue_count": len(missing_columns),
    "level": "FAIL" if missing_columns else "PASS"
})
quality_summary.append({
    "check": "Missing Values",
    "issue_count": total_missing,
    "level": "FAIL" if total_missing > 0 else "PASS"
})
quality_summary.append({
    "check": "Duplicate IDs",
    "issue_count": duplicate_count,
    "level": "FAIL" if duplicate_count > 0 else "PASS"
})
non_numeric_total = int(type_report["non_numeric_count"].sum())
quality_summary.append({
    "check": "Non-numeric Values",
    "issue_count": non_numeric_total,
    "level": "FAIL" if non_numeric_total > 0 else "PASS"
})
range_violation_total = int(range_report["violation_count"].sum())
quality_summary.append({
    "check": "Range Violations",
    "issue_count": range_violation_total,
    "level": "FAIL" if range_violation_total > 0 else "PASS"
})
iqr_outlier_total = int(iqr_report["outlier_count"].sum())
quality_summary.append({
    "check": "IQR Outliers",
    "issue_count": iqr_outlier_total,
    "level": "WARN" if iqr_outlier_total > 0 else "PASS"
})
summary_df = pd.DataFrame(quality_summary)
display(summary_df)

## 셀 18. 전체 데이터 품질 리포트 그래프

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_df["check"], summary_df["issue_count"])
plt.title("Data Quality Report")
plt.xlabel("Quality Check")
plt.ylabel("Issue Count")
plt.xticks(rotation=45)
plt.show()

## 셀 19. PASS / WARN / FAIL 개수 시각화

In [ ]:
status_counts = (
    summary_df["level"]
    .value_counts()
    .reindex(["PASS", "WARN", "FAIL"], fill_value=0)
)
display(status_counts)

plt.figure(figsize=(6, 4))
plt.bar(status_counts.index, status_counts.values)
plt.title("Quality Check Status")
plt.xlabel("Status")
plt.ylabel("Number of Checks")
plt.show()

## 셀 20. 최종 품질 판정

In [ ]:
fail_count = int((summary_df["level"] == "FAIL").sum())
warn_count = int((summary_df["level"] == "WARN").sum())

print("=================================")
print("      DATA QUALITY RESULT")
print("=================================")
if fail_count > 0:
    print("RESULT : FAILED")
    print(f"FAIL 검사 : {fail_count}")
    print(f"WARN 검사 : {warn_count}")
else:
    print("RESULT : PASSED")
    print(f"FAIL 검사 : {fail_count}")
    print(f"WARN 검사 : {warn_count}")
print("=================================")

## 셀 21. 품질 리포트를 CSV로 저장

In [ ]:
summary_df.to_csv("data_quality_report.csv", index=False)
print("data_quality_report.csv 저장 완료")

In [ ]:
from google.colab import files
files.download("data_quality_report.csv")

# 실습 정리

- 전처리: 문제가 있는 데이터를 수정·변환하는 과정
- 품질검증: 전처리 결과가 정해진 기준을 만족하는지 확인하는 과정
- 범위 위반: 명백한 품질 오류 → FAIL
- 통계적 이상치: 실제 데이터일 수 있음 → WARN
- 자동화 환경에서는 품질검사 결과를 Quality Gate로 활용 가능